In [1]:
from dataclasses import dataclass
import torch
from torch import nn
import torch.nn.functional as F
from typing import Tuple, Any, Optional, List
from transformers import GPT2LMHeadModel
import tiktoken

**Creating our GPT model config class:**

In [2]:
@dataclass
class GPTConfig:
    vocab_size: int = 50257  # Vocabulary size
    context_length: int = 1024  # Context length
    emb_dim: int = 768  # Embedding dimension
    n_heads: int = 12  # Number of attention heads
    n_layers: int = 12  # Number of layers
    drop_rate: float = 0.1  # Dropout rate
    qkv_bias: bool = False  # Query-Key-Value bias

---

**LayerNorm**

In [3]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x: torch.Tensor):
        mean = x.mean(dim=-1, keepdim=True) # x shape: (batch_size, seq_length, emb_dim) -> mean shape: (batch_size, seq_length, 1)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        norm_x = (x - mean) / torch.sqrt(var + self.eps) # Normalize to zero mean and variance of 1
        return self.scale * norm_x + self.shift

**LayerNorm** normalizes each token's embedding vector independently.

If the input is:

\(x \in \mathbb{R}^{B \times T \times d_{model}}\)

LayerNorm works on the last dimension (\(d_{model}\)) for each token position \((b, t)\):

$$
\mu_{b,t} = \frac{1}{d_{model}}\sum_{i=1}^{d_{model}} x_{b,t,i}
$$

$$
\sigma^2_{b,t} = \frac{1}{d_{model}}\sum_{i=1}^{d_{model}} (x_{b,t,i} - \mu_{b,t})^2
$$

$$
\hat{x}_{b,t,i} = \frac{x_{b,t,i} - \mu_{b,t}}{\sqrt{\sigma^2_{b,t} + \epsilon}}
$$

$$
\mathrm{LayerNorm}(x_{b,t,i}) = \gamma_i\hat{x}_{b,t,i} + \beta_i
$$

What this means in practice:

- Every token gets zero-mean, unit-variance features before the next sublayer.
- \( \gamma\ ) (`scale`) and \(\beta\) (`shift`) let the model learn the best normalized feature scale.
- Unlike BatchNorm, this does not depend on other examples in the batch.

Why GPT-style models use it:

- Training is more stable.
- Works reliably with small or changing batch sizes.
- Fits sequence modeling, where each token should be normalized independently.


---
**Multi-Head Attention**

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert not (d_out % num_heads), "d_out must be divisible by num_heads because we need to split the output into num_heads different heads for multi head attention"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // self.num_heads

        self.W_query = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_key = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)
        self.W_value = nn.Linear(in_features=d_in, out_features=d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(p=dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )  # masking upper triangular part of the attention scores matrix to prevent attending to future tokens

    def forward(self, x):
        b, num_tokens, d_in = x.shape # x shape: (batch_size, seq_length, emb_dim)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # adding num_heads dimension for efficient multi head mm
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) # keys shape: (batch_size, seq_length, num_heads, head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim) # queries shape: (batch_size, seq_length, num_heads, head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim) # values shape: (batch_size, seq_length, num_heads, head_dim)

        # transposing from shape (b, num_tokens, num_heads, head_dim) to (b, num_heads, num_tokens, head_dim) in order to perform multi head mm efficiently
        keys = keys.transpose(1, 2) # keys shape: (batch_size, num_heads, seq_length, head_dim)
        queries = queries.transpose(1, 2) # queries shape: (batch_size, num_heads, seq_length, head_dim)
        values = values.transpose(1, 2) # values shape: (batch_size, num_heads, seq_length, head_dim)

        attn_scores = queries @ keys.transpose(2, 3) # attn_scores shape: (batch_size, num_heads, seq_length, seq_length)
        mask_bool = self.mask[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf) # masking the upper triangular part of the attention scores matrix to prevent attending to future tokens

        attn_weights = torch.softmax(attn_scores / self.head_dim ** 0.5, dim=-1) # attn_weights shape: (batch_size, num_heads, seq_length, seq_length)

        attn_weights = self.dropout(attn_weights) # applying dropout to attention weights for regularization

        context_vec = (attn_weights @ values).transpose(1, 2) # context_vec shape: (batch_size, num_heads, seq_length, head_dim) -> (batch_size, seq_length, num_heads, head_dim)

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out) # context_vec shape: (batch_size, seq_length, d_out)

        context_vec = self.out_proj(context_vec) # context_vec shape: (batch_size, seq_length, d_out)
        return context_vec

**Multi-Head Attention: what this block computes**

Goal: each token decides which previous tokens are relevant, then mixes information from them.

Let the input be:
$$
(X \in \mathbb{R}^{B \times T \times d_{model}})
$$
Step 1. Build query, key, value features:

$$
Q = XW_Q, \quad K = XW_K, \quad V = XW_V
$$

$$
(W_Q, W_K, W_V \in \mathbb{R}^{d_{model} \times d_{model}}).
$$

Step 2. Split into $ (H) $ heads, with $ (d_h = d_{model}/H) $
$$
(Q, K, V \rightarrow \mathbb{R}^{B \times H \times T \times d_h})
$$
Step 3. Compute attention scores for each head:

$$
S_h = \frac{Q_h K_h^\top}{\sqrt{d_h}} + M
$$

where \(M\) is the causal mask:

$$
M_{ij} = \begin{cases}
0 & j \le i \\
-\infty & j > i
\end{cases}
$$

The mask enforces autoregressive behavior: token \(i\) cannot attend to future tokens \(j>i\).

Step 4. Turn scores into probabilities:

$$
A_h = \mathrm{softmax}(S_h)
$$

Each row of \(A_h\) sums to 1 and represents "how much token \(i\) looks at token \(j\)".

Step 5. Weighted sum of values:

$$
C_h = A_h V_h
$$

Step 6. Concatenate heads and project:

$$
\mathrm{MHA}(X) = \mathrm{Concat}(C_1, \ldots, C_H)W_O
$$

with output shape \(\mathbb{R}^{B \times T \times d_{model}}\).

Why multiple heads help:

- Different heads can learn different relations (syntax, long-range dependency, phrase boundaries, etc.).
- Combined heads give a richer representation than one single attention map.


In [5]:
class FeedForward(nn.Module):
    def __init__(self, gpt_config: GPTConfig):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=gpt_config.emb_dim, out_features=4 * gpt_config.emb_dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(in_features=4 * gpt_config.emb_dim, out_features=gpt_config.emb_dim),
        )

    def forward(self, x):
        return self.layers(x)

---
**Feed-Forward Network (MLP) per token**

After attention mixes information across tokens, the FFN transforms each token representation independently.

For a token vector $$ (x \in \mathbb{R}^{d_{model}}): $$

$$
z = W_1x + b_1
$$

$$
h = \mathrm{GELU}(z)
$$

$$
\mathrm{FFN}(x) = W_2h + b_2
$$

Combined form:

$$
\mathrm{FFN}(x) = W_2\,(\mathrm{GELU}(W_1x + b_1)) + b_2
$$

Parameter shapes:

$$
W_1 \in \mathbb{R}^{d_{model} \times 4d_{model}}, \quad
W_2 \in \mathbb{R}^{4d_{model} \times d_{model}}
$$

Key intuition:

- First layer expands feature space $d_{model} \to 4 \times d_{model}$.
- GELU adds non-linearity.
- Second layer compresses back to $ d_{model} $.
- Attention handles token-to-token interaction; FFN handles per-token feature transformation.


In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, gpt_config: GPTConfig):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=gpt_config.emb_dim,
            d_out=gpt_config.emb_dim,
            context_length=gpt_config.context_length,
            dropout=gpt_config.drop_rate,
            num_heads=gpt_config.n_heads,
            qkv_bias=gpt_config.qkv_bias,
        )
        self.ff = FeedForward(gpt_config=gpt_config)
        self.norm1 = LayerNorm(emb_dim=gpt_config.emb_dim)
        self.norm2 = LayerNorm(emb_dim=gpt_config.emb_dim)
        self.drop_shortcut = nn.Dropout(p=gpt_config.drop_rate)

    def forward(self, x):

        shortcut = x  # residual connection for attention
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x += shortcut

        shortcut = x  # residual connection for ff bloc
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x += shortcut

        return x

---
**Transformer Block (Pre-Norm + Residuals)**

A GPT block has two sublayers: attention, then FFN. This implementation is pre-norm (LayerNorm before each sublayer).

Given input \(X\):

1. Attention sublayer:

$$
H = X + \mathrm{Dropout}(\mathrm{MHA}(\mathrm{LN}_1(X)))
$$

2. Feed-forward sublayer:

$$
Y = H + \mathrm{Dropout}(\mathrm{FFN}(\mathrm{LN}_2(H)))
$$

Why this structure works well:

- Residual connections (`+ X`, `+ H`) help gradient flow through deep stacks.
- Pre-norm improves stability for large/deep transformers.
- Dropout regularizes each sublayer output.
- Input and output shapes stay the same: \(\mathbb{R}^{B \times T \times d_{model}}\), so blocks can be stacked easily.


In [7]:
class GPTModel(nn.Module):
    def __init__(self, gpt_config: GPTConfig):
        super().__init__()
        self.tok_emb = nn.Embedding(num_embeddings=gpt_config.vocab_size, embedding_dim=gpt_config.emb_dim)
        self.pos_emb = nn.Embedding(num_embeddings=gpt_config.context_length, embedding_dim=gpt_config.emb_dim)
        self.drop_emb = nn.Dropout(gpt_config.drop_rate)

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(gpt_config=gpt_config) for _ in range(gpt_config.n_layers)]
        )

        self.final_norm = LayerNorm(emb_dim=gpt_config.emb_dim)
        self.out_head = nn.Linear(
            in_features=gpt_config.emb_dim, out_features=gpt_config.vocab_size, bias=False
        )

        # print(f"shape of output head weights before tying: {self.out_head.weight.shape}, shape of token embedding weights: {self.tok_emb.weight.shape}")
        self.out_head.weight = self.tok_emb.weight # tying the output head weights with the token embedding weights
        # print(f"shape of output head weights after tying: {self.out_head.weight.shape}, shape of token embedding weights: {self.tok_emb.weight.shape}")



    def forward(self, in_idx: torch.Tensor):
        batch_size, seq_len = in_idx.shape
        in_idx = in_idx.type(dtype=torch.long)
        tok_embeds = self.tok_emb(in_idx)

        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

---
**GPTModel: from token IDs to next-token logits**

Suppose the token IDs are $$ ((t_1, \ldots, t_T)) $$

Step 1. Token and positional embeddings:

$$
X_0 = E_{tok}[t] + E_{pos}[0:T]
$$

where both terms have shape  $$ (\mathbb{R}^{B \times T \times d_{model}}) $$

Step 2. Pass through \(L\) transformer blocks:

$$
X_\ell = \mathrm{Block}_\ell(X_{\ell-1}), \quad \ell=1,\ldots,L
$$

Step 3. Final normalization + vocabulary projection:

$$
\mathrm{logits} = \mathrm{LN}_f(X_L)W_{out}
$$

Interpretation:

- At position \(t\), the logits vector scores every possible next token.
- Next-token probabilities come from softmax:

$$
p(x_{t+1}\mid x_{\le t}) = \mathrm{softmax}(\mathrm{logits}_t)
$$

Training objective (language modeling):

$$
\mathcal{L} = -\sum_t \log p(x_t \mid x_{<t})
$$

This notebook ties output and token-embedding weights:

$$
W_{out} = E_{tok}^\top
$$

Weight tying reduces parameters and is standard in GPT-style models.


In [8]:
def load_hf_gpt2_weights(custom_model: GPTModel, model_name="gpt2"):
    """
    Loads Hugging Face GPT-2 weights into the custom GPTModel architecture.
    """
    print(f"Loading '{model_name}' weights from Hugging Face...")

    # 1. Load the official Hugging Face model
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    hf_sd = hf_model.state_dict()
    custom_sd = custom_model.state_dict()

    # 2. Extract configuration constants
    n_layers = custom_model.trf_blocks.__len__()
    emb_dim = custom_model.tok_emb.embedding_dim

    with torch.no_grad():
        # --- Embeddings & Final LayerNorm ---
        # Token and Positional Embeddings
        custom_model.tok_emb.weight.copy_(hf_sd["transformer.wte.weight"])
        custom_model.pos_emb.weight.copy_(hf_sd["transformer.wpe.weight"])

        # Tie the LM head weights to the token embeddings
        custom_model.out_head.weight.copy_(hf_sd["transformer.wte.weight"])

        # Final LayerNorm (Custom model uses 'scale' and 'shift' instead of 'weight' and 'bias')
        custom_model.final_norm.scale.copy_(hf_sd["transformer.ln_f.weight"])
        custom_model.final_norm.shift.copy_(hf_sd["transformer.ln_f.bias"])

        # --- Transformer Blocks ---
        for b in range(n_layers):
            # 1. Block LayerNorms
            custom_model.trf_blocks[b].norm1.scale.copy_(hf_sd[f"transformer.h.{b}.ln_1.weight"])
            custom_model.trf_blocks[b].norm1.shift.copy_(hf_sd[f"transformer.h.{b}.ln_1.bias"])

            custom_model.trf_blocks[b].norm2.scale.copy_(hf_sd[f"transformer.h.{b}.ln_2.weight"])
            custom_model.trf_blocks[b].norm2.shift.copy_(hf_sd[f"transformer.h.{b}.ln_2.bias"])

            # 2. Attention (Fused QKV -> Separate Q, K, V)
            # HF shape: (emb_dim, 3 * emb_dim). We transpose to (3 * emb_dim, emb_dim) and split.
            c_attn_weight = hf_sd[f"transformer.h.{b}.attn.c_attn.weight"].t()
            q_w, k_w, v_w = c_attn_weight.split(emb_dim, dim=0)

            custom_model.trf_blocks[b].att.W_query.weight.copy_(q_w)
            custom_model.trf_blocks[b].att.W_key.weight.copy_(k_w)
            custom_model.trf_blocks[b].att.W_value.weight.copy_(v_w)

            # Split biases for Q, K, V
            c_attn_bias = hf_sd[f"transformer.h.{b}.attn.c_attn.bias"]
            q_b, k_b, v_b = c_attn_bias.split(emb_dim, dim=0)

            custom_model.trf_blocks[b].att.W_query.bias.copy_(q_b)
            custom_model.trf_blocks[b].att.W_key.bias.copy_(k_b)
            custom_model.trf_blocks[b].att.W_value.bias.copy_(v_b)

            # Attention output projection
            c_proj_weight = hf_sd[f"transformer.h.{b}.attn.c_proj.weight"].t()
            custom_model.trf_blocks[b].att.out_proj.weight.copy_(c_proj_weight)
            custom_model.trf_blocks[b].att.out_proj.bias.copy_(hf_sd[f"transformer.h.{b}.attn.c_proj.bias"])

            # 3. Feed Forward Network (MLP)
            # HF uses c_fc for the first layer and c_proj for the second
            mlp_c_fc_weight = hf_sd[f"transformer.h.{b}.mlp.c_fc.weight"].t()
            custom_model.trf_blocks[b].ff.layers[0].weight.copy_(mlp_c_fc_weight)
            custom_model.trf_blocks[b].ff.layers[0].bias.copy_(hf_sd[f"transformer.h.{b}.mlp.c_fc.bias"])

            mlp_c_proj_weight = hf_sd[f"transformer.h.{b}.mlp.c_proj.weight"].t()
            custom_model.trf_blocks[b].ff.layers[2].weight.copy_(mlp_c_proj_weight)
            custom_model.trf_blocks[b].ff.layers[2].bias.copy_(hf_sd[f"transformer.h.{b}.mlp.c_proj.bias"])

    print("Weights successfully loaded!")
    return custom_model

---
**How Hugging Face GPT-2 weights are mapped**

Your custom model and Hugging Face GPT-2 are mathematically equivalent, but parameter layout differs.

Attention in Hugging Face uses one fused projection:

$$
[Q\;K\;V] = XW_{c\_attn} + b_{c\_attn}
$$

with:

$$
W_{c\_attn} \in \mathbb{R}^{d_{model} \times 3d_{model}}, \quad b_{c\_attn} \in \mathbb{R}^{3d_{model}}
$$

In this notebook, attention uses three separate layers:

$$
Q = XW_Q + b_Q, \quad K = XW_K + b_K, \quad V = XW_V + b_V
$$

So loading requires:

- Splitting fused GPT-2 weights/biases into 3 chunks (Q, K, V).
- Transposing matrices where PyTorch module conventions differ.
- Copying block-by-block for LayerNorm, attention projections, and MLP projections.

After mapping, the model should produce GPT-2-compatible outputs for the same tokenizer and prompts.


In [9]:
model_config = GPTConfig(qkv_bias=True)
gpt2_model = GPTModel(gpt_config=model_config)

In [10]:
with torch.inference_mode():
    print(gpt2_model(torch.rand(12).unsqueeze(dim=0)).shape)


torch.Size([1, 12, 50257])


In [12]:

# 3. Load the weights
gpt2_model = load_hf_gpt2_weights(custom_model=gpt2_model, model_name="gpt2")




Loading 'gpt2' weights from Hugging Face...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Weights successfully loaded!


In [16]:
def generate_text(model: GPTModel, tokenizer: tiktoken.Encoding, text: str, max_new_tokens: int, context_length: int, temperature: float = 1.0, top_k: int = None, device: str = "cpu"):
    encoded = tokenizer.encode(text)
    tokenized = tokenizer.decode_tokens_bytes(encoded)
    print(f"tokenized input: {tokenized} | number of tokens: {len(encoded)}")
    print(f"encoded input: {encoded}")

    in_idx = torch.tensor(encoded).unsqueeze(0).to(device) # Shape: (1, seq_len)

    model.eval()
    for _ in range(max_new_tokens):
        in_idx_cond = in_idx[:, -context_length:] # Crop context to the model's maximum context length

        with torch.inference_mode():
            logits = model(in_idx_cond)

        # 1. Focus on the last step and apply temperature
        logits = logits[:, -1, :] / temperature

        # 2. Optional: Top-K filtering
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        # 3. Convert to probabilities via Softmax
        probs = torch.softmax(logits, dim=-1)

        # 4. Sample from the distribution (instead of argmax)
        next_token = torch.multinomial(probs, num_samples=1)

        in_idx = torch.cat((in_idx, next_token), dim=1)

        # Stop if we hit end-of-text token
        if next_token.item() == tokenizer.eot_token:
            break

    return tokenizer.decode(in_idx.squeeze(0).tolist())

---
**Autoregressive generation with temperature and top-k**

Generation predicts one token at a time, appends it, and repeats.

For current context tokens $$ ((x_1,\ldots,x_t)) $$

1. Run model and take only last-position logits \(z\).
2. Apply temperature $ (\tau) $:

$$
\hat{z}_i = z_i / \tau
$$

- $(\tau < 1)$: sharper, more deterministic.
- $(\tau > 1)$: flatter, more diverse.

3. Optional top-k filter:

$$
\tilde{z}_i = \begin{cases}
\hat{z}_i & i \in \mathrm{TopK}(\hat{z}) \\
-\infty & \text{otherwise}
\end{cases}
$$

4. Convert to probabilities and sample:

$$
p = \mathrm{softmax}(\tilde{z})
$$

5. Append $(x_{t+1})$ to context and continue until:

- `end-of-text token` is sampled, or
- `max_new_tokens` is reached.


In [13]:
tokenizer = tiktoken.get_encoding("gpt2")

In [14]:
prompt = "Openchip hands-on workshops are awesome because"

In [18]:
output = generate_text(
    model=gpt2_model,
    tokenizer=tokenizer,
    text=prompt,
    max_new_tokens=64,
    temperature=1.2,
    top_k=15,
    context_length=model_config.context_length
)

print(f"\nGenerated text:\n{output}")

tokenized input: [b'Open', b'chip', b' hands', b'-', b'on', b' workshops', b' are', b' awesome', b' because'] | number of tokens: 9
encoded input: [11505, 35902, 2832, 12, 261, 25982, 389, 7427, 780]

Generated text:
Openchip hands-on workshops are awesome because people get to learn and be creative. They get more fun with their hands on workshops, so it makes the process of teaching and learning so more fun!

So, for this month's workshop: I am excited about this upcoming workshop that I am working on called "Giant Hands-on." It will be
